# Chess Learning — Stage 02 (Stockfish Teacher, Neural Net): TD Learning vs Stockfish

Trains the NN evaluator by playing games against Stockfish at a configurable ELO,
rather than against itself. All positions in each game (both engine and Stockfish
half-moves) are used in the TD chain.

| Component | Description |
|---|---|
| **Engine** | `engine_nn.py` — ChessNet MLP (19→32→1, tanh) |
| **Opponent** | Stockfish at configurable ELO (minimum 1320 for Stockfish 19) |
| **Learning** | TD(0) + Adam; per-position online updates on all positions in the game |
| **No epsilon-greedy** | Stockfish provides positional diversity; engine plays best move |
| **Logging** | Attribution weights (dV/dxᵢ at starting position) at each checkpoint |

**Research question:** Does a calibrated external teacher stabilise NN weight learning
and halt the ELO decline seen in pure self-play (run_id=24)?

In [12]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "chess", "-q"])

0

In [13]:
try:
    import torch
    print(f"PyTorch {torch.__version__} already available")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "-q"])
    import torch
    print(f"PyTorch {torch.__version__} installed")

PyTorch 2.14.1+cpu already available


In [14]:
import asyncio
import chess
import chess.engine
import chess.pgn
import glob
import os
import sqlite3
import time
import torch
from collections import defaultdict
from datetime import datetime

### Import engine

Uses `engine_nn.py`. Feature extraction is shared with `engine_learnable_material.py`.

In [15]:
sys.path.insert(0, os.path.dirname(os.path.abspath("selfplay_sf_nn.ipynb")))
from engine_nn import (
    FEATURE_NAMES, N_FEATURES,
    new_model, save_model, load_model,
    attribution_weights,
    extract_features,
    get_best_move,
)
print(f"Engine loaded. Features ({N_FEATURES}):", FEATURE_NAMES)
print("PyTorch version:", torch.__version__)

Engine loaded. Features (19): ['material_queen', 'material_rook', 'material_bishop', 'material_knight', 'material_pawn', 'passed_pawn', 'doubled_pawn', 'isolated_pawn', 'backward_pawn', 'king_safety', 'center_control', 'rook_open_file', 'connected_rooks', 'knight_outpost', 'bishop_pair', 'rook_seventh', 'piece_development', 'mobility', 'pawn_advancement']
PyTorch version: 2.14.1+cpu


## Database Setup

Identical schema to `selfplay_nn.ipynb`. R analysis scripts work unchanged.

In [16]:
def setup_database(db_path):
    os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
    conn = sqlite3.connect(db_path)
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS runs (
            run_id         INTEGER PRIMARY KEY,
            started_at     TEXT,
            depth          INTEGER,
            learning_rate  REAL,
            move_limit     INTEGER,
            notes          TEXT,
            engine_version TEXT
        );
        CREATE TABLE IF NOT EXISTS games (
            game_id          INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id           INTEGER,
            game_number      INTEGER,
            outcome          TEXT,
            n_halfmoves      INTEGER,
            duration_s       REAL,
            terminated_by    TEXT,
            seed_run_id      INTEGER,
            seed_game_number INTEGER
        );
        CREATE TABLE IF NOT EXISTS weights (
            weight_id    INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id       INTEGER,
            game_number  INTEGER,
            feature_name TEXT,
            weight_value REAL
        );
        CREATE TABLE IF NOT EXISTS weight_deltas (
            delta_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id       INTEGER,
            game_number  INTEGER,
            feature_name TEXT,
            delta        REAL
        );
        CREATE TABLE IF NOT EXISTS feature_stats (
            stat_id      INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id       INTEGER,
            game_number  INTEGER,
            outcome      TEXT,
            feature_name TEXT,
            mean_value   REAL
        );
        CREATE TABLE IF NOT EXISTS game_records (
            record_id   INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id      INTEGER,
            game_number INTEGER,
            pgn         TEXT,
            td_error    REAL
        );
        CREATE TABLE IF NOT EXISTS position_logs (
            pos_id        INTEGER PRIMARY KEY AUTOINCREMENT,
            run_id        INTEGER,
            game_number   INTEGER,
            ply           INTEGER,
            feature_name  TEXT,
            feature_value REAL
        );
    """)
    conn.commit()
    return conn


def log_run(conn, run_id, depth, lr, move_limit, notes, engine_version="v2_neural_net_h32"):
    conn.execute(
        "INSERT OR IGNORE INTO runs "
        "(run_id, started_at, depth, learning_rate, move_limit, notes, engine_version) "
        "VALUES (?,?,?,?,?,?,?)",
        (run_id, datetime.now().isoformat(), depth, lr, move_limit, notes, engine_version),
    )
    conn.commit()


def log_game(conn, run_id, game_num, outcome, n_halfmoves, duration_s, terminated_by):
    conn.execute(
        "INSERT INTO games "
        "(run_id, game_number, outcome, n_halfmoves, duration_s, terminated_by) "
        "VALUES (?,?,?,?,?,?)",
        (run_id, game_num, outcome, n_halfmoves, duration_s, terminated_by),
    )
    conn.commit()


def log_weights(conn, run_id, game_num, weights_dict):
    conn.executemany(
        "INSERT INTO weights (run_id, game_number, feature_name, weight_value) VALUES (?,?,?,?)",
        [(run_id, game_num, name, val) for name, val in weights_dict.items()],
    )
    conn.commit()


def log_weight_deltas(conn, run_id, game_num, deltas_dict):
    conn.executemany(
        "INSERT INTO weight_deltas (run_id, game_number, feature_name, delta) VALUES (?,?,?,?)",
        [(run_id, game_num, name, val) for name, val in deltas_dict.items()],
    )
    conn.commit()


def log_game_record(conn, run_id, game_num, pgn, td_error):
    conn.execute(
        "INSERT INTO game_records (run_id, game_number, pgn, td_error) VALUES (?,?,?,?)",
        (run_id, game_num, pgn, td_error),
    )
    conn.commit()


def log_feature_stats(conn, run_id, game_num, positions, outcome):
    white_feats = defaultdict(list)
    for feat_dict, color in positions:
        sign = 1.0 if color == chess.WHITE else -1.0
        for name, val in feat_dict.items():
            white_feats[name].append(sign * val)
    rows = [
        (run_id, game_num, outcome, name, sum(vals) / len(vals))
        for name, vals in white_feats.items() if vals
    ]
    conn.executemany(
        "INSERT INTO feature_stats "
        "(run_id, game_number, outcome, feature_name, mean_value) VALUES (?,?,?,?,?)",
        rows,
    )
    conn.commit()


def log_position_logs(conn, run_id, game_num, positions):
    rows = []
    for ply, (feat_dict, color) in enumerate(positions):
        for name, val in feat_dict.items():
            rows.append((run_id, game_num, ply, name, val))
    conn.executemany(
        "INSERT INTO position_logs "
        "(run_id, game_number, ply, feature_name, feature_value) VALUES (?,?,?,?,?)",
        rows,
    )
    conn.commit()


def get_last_game_number(conn, run_id):
    row = conn.execute(
        "SELECT MAX(game_number) FROM games WHERE run_id = ?", (run_id,)
    ).fetchone()
    return row[0] if row[0] is not None else 0


def load_last_model(run_id, model_dir):
    """Load the most recent nn_sf_run checkpoint for this run_id."""
    pattern = os.path.join(model_dir, f"nn_sf_run_{run_id}_game_*.pt")
    files   = glob.glob(pattern)
    if not files:
        raise FileNotFoundError(
            f"No SF-teacher checkpoint files found for run_id={run_id} in {model_dir}"
        )
    latest = max(
        files,
        key=lambda p: int(p.replace('\\', '/').split('_game_')[-1].replace('.pt', ''))
    )
    print(f"Loading checkpoint: {latest}")
    return load_model(latest)

## Stockfish Setup

Auto-detects Stockfish installed via `winget install Stockfish.Stockfish`.
Minimum ELO for Stockfish 19 is 1320.

In [17]:
def find_stockfish():
    local_app = os.environ.get("LOCALAPPDATA", "")
    pattern   = os.path.join(
        local_app, "Microsoft", "WinGet", "Packages", "Stockfish*", "**", "stockfish*.exe"
    )
    hits = glob.glob(pattern, recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(
        "Stockfish not found. Install with: winget install Stockfish.Stockfish"
    )


SF_PATH = find_stockfish()
print(f"Stockfish found: {SF_PATH}")

# Windows: Jupyter uses SelectorEventLoop which does not support subprocess transport.
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    print("Windows ProactorEventLoop set.")

Stockfish found: C:\Users\dkill\AppData\Local\Microsoft\WinGet\Packages\Stockfish.Stockfish_Microsoft.Winget.Source_8wekyb3d8bbwe\stockfish\stockfish-windows-x86-64-universal.exe
Windows ProactorEventLoop set.


## Game Function

Engine plays at `DEPTH` using the NN evaluator; Stockfish uses its own search at
the configured ELO. All positions are recorded as `(feature_dict, color)` 2-tuples
so the full TD chain is intact across all half-moves.

Engine color alternates across games: odd game numbers = White, even = Black.

In [18]:
def play_game_vs_sf_nn(sf, model, depth, engine_color):
    """
    Play one game: NN engine (engine_color) vs Stockfish.

    sf           : open SimpleEngine, pre-configured with UCI_Elo limit.
    model        : ChessNet instance.
    depth        : negamax search depth for the engine.
    engine_color : chess.WHITE or chess.BLACK.

    Games terminate under standard chess rules only (claim_draw=True):
    checkmate, stalemate, insufficient material, 50-move rule,
    threefold repetition, 75-move rule. No artificial move limit.

    Returns: positions, outcome, terminated_by, pgn_str
    positions : list of (feature_dict, color) 2-tuples for every half-move.
    """
    board         = chess.Board()
    game          = chess.pgn.Game()
    node          = game
    positions     = []
    terminated_by = "natural"

    while True:
        if board.is_game_over(claim_draw=True):
            terminated_by = "natural"
            break

        features = extract_features(board)
        color    = board.turn

        if board.turn == engine_color:
            move, _ = get_best_move(board, model, depth=depth, randomize=False)
        else:
            result = sf.play(board, chess.engine.Limit(time=0.5), ponder=False)
            move   = result.move

        if move is None:
            break

        node = node.add_variation(move)
        board.push(move)
        positions.append((features, color))

    outcome = board.result(claim_draw=True)

    game.headers["Result"] = outcome
    return positions, outcome, terminated_by, str(game)

## TD(0) Update (Neural Net)

All positions in the game receive gradient updates — no `is_learner` filtering.
One Adam step per position, applied online. Returns updated attribution weights
and per-feature deltas for logging.

In [19]:
def outcome_for_color(outcome, color):
    if outcome == "1-0":
        return  1.0 if color == chess.WHITE else -1.0
    elif outcome == "0-1":
        return -1.0 if color == chess.WHITE else  1.0
    return 0.0


def td_update_sf_nn(model, optimizer, positions, outcome, draw_lr_scale=0.1):
    """
    Online TD(0) update for the NN evaluator vs Stockfish.

    positions : list of (feature_dict, color) 2-tuples from play_game_vs_sf_nn().
    All positions receive gradient updates (no is_learner filtering).

    Returns: (attr_weights_dict, deltas_dict, mean_abs_td_error)
    """
    is_draw = (outcome == "1/2-1/2")
    scale   = draw_lr_scale if is_draw else 1.0

    tensors = [
        torch.tensor([feat[name] for name in FEATURE_NAMES], dtype=torch.float32)
        for feat, _ in positions
    ]

    # Bootstrap targets: fixed constants (no grad)
    with torch.no_grad():
        next_values = [model(x) for x in tensors[1:]]

    attr_before = attribution_weights(model)

    total_error = 0.0
    for t, (feat, color) in enumerate(positions):
        if t < len(positions) - 1:
            target = -next_values[t]          # perspective flip
        else:
            target = torch.tensor(outcome_for_color(outcome, color))

        optimizer.zero_grad()
        v    = model(tensors[t])
        loss = scale * (target - v) ** 2
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            total_error += abs((target - v).item())

    attr_after = attribution_weights(model)
    deltas     = {name: attr_after[name] - attr_before[name] for name in FEATURE_NAMES}
    td_error   = total_error / len(positions) if positions else 0.0

    return attr_after, deltas, td_error

## Training Loop

Opens one Stockfish process for the full session and closes it safely via `try/finally`.

In [20]:
def train_vs_sf_nn(
    run_id,
    n_games,
    sf_elo           = 1320,
    depth            = 3,
    learning_rate    = 0.001,
    checkpoint_every = 10,
    db_path          = "../../data/chess_learning.db",
    model_dir        = "../../models",
    notes            = "",
    resume           = False,
    hidden_size      = 32,
    draw_lr_scale    = 0.1,
    log_positions    = False,
    verbose          = True,
):
    conn = setup_database(db_path)
    os.makedirs(model_dir, exist_ok=True)

    if resume:
        model      = load_last_model(run_id, model_dir)
        start_game = get_last_game_number(conn, run_id) + 1
        print(f"Resuming run_id={run_id} from game {start_game}")
    else:
        model      = new_model(hidden_size=hidden_size)
        start_game = 1
        log_run(conn, run_id, depth, learning_rate, None, notes)
        log_weights(conn, run_id, 0, attribution_weights(model))

    model.eval()   # eval mode during play; Adam still updates parameters
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    outcomes = {"1-0": 0, "0-1": 0, "1/2-1/2": 0}
    t_start  = time.time()
    attr_w   = attribution_weights(model)

    sf = chess.engine.SimpleEngine.popen_uci(SF_PATH)
    sf.configure({"UCI_LimitStrength": True, "UCI_Elo": sf_elo})
    print(f"Stockfish open at ELO {sf_elo}. Starting run_id={run_id} from game {start_game}.")

    try:
        for i in range(n_games):
            game_num     = start_game + i
            engine_color = chess.WHITE if (game_num % 2 == 1) else chess.BLACK
            color_label  = "White" if engine_color == chess.WHITE else "Black"
            t0           = time.time()

            positions, outcome, terminated_by, pgn = play_game_vs_sf_nn(
                sf, model, depth, engine_color
            )
            elapsed = time.time() - t0

            attr_w, deltas, td_error = td_update_sf_nn(
                model, optimizer, positions, outcome, draw_lr_scale
            )

            log_game(conn, run_id, game_num, outcome, len(positions), elapsed, terminated_by)
            log_game_record(conn, run_id, game_num, pgn, td_error)
            log_weight_deltas(conn, run_id, game_num, deltas)
            log_feature_stats(conn, run_id, game_num, positions, outcome)
            if log_positions:
                log_position_logs(conn, run_id, game_num, positions)

            outcomes[outcome] = outcomes.get(outcome, 0) + 1

            if game_num % checkpoint_every == 0:
                log_weights(conn, run_id, game_num, attr_w)
                pt_path = os.path.join(model_dir, f"nn_sf_run_{run_id}_game_{game_num}.pt")
                save_model(model, pt_path)
                if verbose:
                    elapsed_total = time.time() - t_start
                    rate  = (i + 1) / elapsed_total * 3600
                    dec   = outcomes.get("1-0", 0) + outcomes.get("0-1", 0)
                    total = sum(outcomes.values())
                    print(
                        f"Game {game_num:5d} ({color_label}) | {outcome:7s} | "
                        f"{len(positions):3d} half-moves | {elapsed:.1f}s | "
                        f"td_err={td_error:.3f} | decisive={dec}/{total} | {rate:.0f} games/hr"
                    )
    finally:
        sf.quit()
        print("Stockfish closed.")

    final_game = start_game + n_games - 1
    if final_game % checkpoint_every != 0:
        log_weights(conn, run_id, final_game, attr_w)
        pt_final = os.path.join(model_dir, f"nn_sf_run_{run_id}_game_{final_game}.pt")
        save_model(model, pt_final)
    else:
        pt_final = os.path.join(model_dir, f"nn_sf_run_{run_id}_game_{final_game}.pt")
    conn.close()
    print(f"\nDone. {n_games} games. Final checkpoint: {pt_final}")
    print(f"Outcomes: {outcomes}")
    return model

## Configuration & Run

Set `RESUME = False` to start a new run; `True` to continue.

**Speed note:** Stockfish uses 0.5s per move; each game takes ~60–120s.
A 50-game session is roughly 1–2 hours.

In [21]:
RUN_ID           = 27          # new run_id for SF-teacher NN condition
N_GAMES          = 200          # games this session
SF_ELO           = 1320        # Stockfish ELO (minimum 1320 for Stockfish 19)
DEPTH            = 3
LEARNING_RATE    = 0.001       # Adam LR
CHECKPOINT_EVERY = 10
HIDDEN_SIZE      = 32          # 19 -> 32 -> 1
DRAW_LR_SCALE    = 0.1
DB_PATH          = "../../data/chess_learning.db"
MODEL_DIR        = "../../models"
RESUME           = True

model = train_vs_sf_nn(
    run_id           = RUN_ID,
    n_games          = N_GAMES,
    sf_elo           = SF_ELO,
    depth            = DEPTH,
    learning_rate    = LEARNING_RATE,
    checkpoint_every = CHECKPOINT_EVERY,
    db_path          = DB_PATH,
    model_dir        = MODEL_DIR,
    resume           = RESUME,
    hidden_size      = HIDDEN_SIZE,
    draw_lr_scale    = DRAW_LR_SCALE,
    notes            = (
        f"SF teacher (NN); SF_ELO={SF_ELO}; 19 features; MLP 19->32->1; "
        f"TD(0)+Adam; depth={DEPTH}; lr={LEARNING_RATE}; no move limit (chess rules only)"
    ),
)

Loading checkpoint: ../../models\nn_sf_run_27_game_440.pt
Resuming run_id=27 from game 441
Stockfish open at ELO 1320. Starting run_id=27 from game 441.
Game   450 (Black) | 0-1     |  92 half-moves | 143.6s | td_err=0.041 | decisive=10/10 | 43 games/hr
Game   460 (Black) | 0-1     |  88 half-moves | 119.9s | td_err=0.041 | decisive=17/20 | 42 games/hr
Game   470 (Black) | 1/2-1/2 |  92 half-moves | 91.1s | td_err=0.057 | decisive=24/30 | 40 games/hr
Game   480 (Black) | 1/2-1/2 |  72 half-moves | 60.5s | td_err=0.052 | decisive=31/40 | 41 games/hr
Game   490 (Black) | 1-0     | 101 half-moves | 72.5s | td_err=0.042 | decisive=40/50 | 37 games/hr
Game   500 (Black) | 0-1     |  76 half-moves | 99.5s | td_err=0.032 | decisive=49/60 | 37 games/hr
Game   510 (Black) | 1-0     |  59 half-moves | 98.7s | td_err=0.049 | decisive=58/70 | 36 games/hr
Game   520 (Black) | 1-0     | 119 half-moves | 118.5s | td_err=0.063 | decisive=65/80 | 36 games/hr
Game   530 (Black) | 1/2-1/2 |  88 half-move

## Attribution Weight Trajectory

Quick check after training. Full visualization in `03_visualize.qmd`.

In [22]:
conn = sqlite3.connect(DB_PATH)
rows = conn.execute(
    "SELECT game_number, feature_name, weight_value FROM weights "
    "WHERE run_id = ? ORDER BY game_number, feature_name",
    (RUN_ID,)
).fetchall()
conn.close()

by_game = defaultdict(dict)
for game_num, feature, val in rows:
    by_game[game_num][feature] = val

game_nums = sorted(by_game.keys())
header = f"{'Attribution (dV/dx_i)':<24}" + "".join(f"{g:>10}" for g in game_nums)
print(header)
print("-" * len(header))
for feature in FEATURE_NAMES:
    row = f"{feature:<24}" + "".join(f"{by_game[g].get(feature, 0):>10.4f}" for g in game_nums)
    print(row)

if game_nums:
    n_correct = sum(1 for f in FEATURE_NAMES if by_game[max(game_nums)].get(f, 0) > 0)
    print(f"\nSign correct (all positive = correct in v2): {n_correct} / {len(FEATURE_NAMES)}")

Attribution (dV/dx_i)            0        10        20        30        40        50        60        70        80        90       100       110       120       130       140       150       160       170       180       190       200       210       220       230       240       250       260       270       280       290       300       310       320       330       340       350       360       370       380       390       400       410       420       430       440       450       460       470       480       490       500       510       520       530       540       550       560       570       580       590       600       610       620       630       640
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------